# 07 · End-to-end LLM (paper §3)

The patient record is the question; the answer is one token per time unit (`[N]` no event,
`[C]` censored, `[Ek]` event k). Three backbones x three training objectives:

* backbones: **GPT-2** (124M, pretrained), **Qwen2.5-0.5B** (pretrained, LoRA), and a small
  GPT-2 trained **from scratch** on the records;
* objectives: **ce** - token cross-entropy (Eq. 8); **deephit** - the DeepHit-style
  likelihood the paper proposes; **deephit+illegal** - plus the illegal-sequence penalty.

Curves are read from the answer-token probabilities and scored with exactly the metrics
of every other model. A T4 GPU is needed (fp16).

## Options

In [ ]:
NAME = "07_llm_end_to_end"

SEEDS = list(range(10))   # protocol: 10 random 60/10/30 splits (as SurvTRACE); seed s = split s and init s

DATASETS = ["metabric", "support", "ebmt"]
BACKBONES = {
    "gpt2": ["llm_init=pretrained", "llm_model=gpt2", "llm_learning_rate=5e-5"],
    "qwen0.5b": ["llm_init=pretrained", "llm_model=Qwen/Qwen2.5-0.5B", "llm_lora_r=16",
                 "llm_learning_rate=2e-4", "llm_batch_size=16", "llm_grad_accum=2"],
    "scratch": ["llm_init=scratch", "llm_model=gpt2", "llm_learning_rate=5e-4"],
}
OBJECTIVES = {"ce": ["llm_loss=ce"], "deephit": ["llm_loss=deephit"],
              "deephit+illegal": ["llm_loss=deephit", "llm_illegal_coeff=1.0"]}
LLM_EPOCHS = 20

# --- execution ------------------------------------------------------------------------
SMOKE_TEST   = False    # True: 2 epochs, 1 seed, 1-2 configurations - only to check that it runs
SMOKE_EPOCHS = 2
WORKERS      = None     # parallel runs; None = 2 on Kaggle (4 vCPU), 4 locally
TIME_BUDGET_MIN = 11 * 60   # stop launching new runs after this (Kaggle sessions end at 12 h);
                            # re-run with this notebook's output attached to continue
INSTALL_DEPS = True     # Kaggle only
GIT_URL      = "https://github.com/Parsagh05/dmmst.git"   # the code is cloned from here
CLEAN_UP_AT_END = True  # Kaggle: delete the working repo copy after the report

## Setup (identical in every notebook)

In [ ]:
# ---- 1. get the code --------------------------------------------------------------
import os, sys, json, shutil, subprocess, time, hashlib
from pathlib import Path

ON_KAGGLE = Path("/kaggle/working").is_dir()

def _is_repo(p):
    return (p / "sat" / "finetune.py").is_file() and (p / "conf").is_dir()

if ON_KAGGLE:
    WORK = Path("/kaggle/working")
    REPO = WORK / "dmmst"
    if not _is_repo(REPO):
        subprocess.run(["git", "clone", "--depth", "1", GIT_URL, str(REPO)], check=True)
    RESULTS_ROOT = WORK / "results"
    INPUT_ROOTS = [Path("/kaggle/input")]
else:
    REPO = Path.cwd().resolve()
    while not _is_repo(REPO) and REPO != REPO.parent:
        REPO = REPO.parent
    RESULTS_ROOT = REPO / "results" / "version_2"
    INPUT_ROOTS = []
assert _is_repo(REPO), REPO
os.chdir(REPO)
sys.path.insert(0, str(REPO))
CODE_FINGERPRINT = hashlib.md5(b"".join(
    p.read_bytes() for p in sorted((REPO / "sat").rglob("*.py")) + sorted((REPO / "conf").rglob("*.yaml"))
)).hexdigest()[:10]
COMMIT = subprocess.run(["git", "rev-parse", "--short", "HEAD"], cwd=REPO, text=True,
                        capture_output=True).stdout.strip()
print("repo:", REPO, "| commit:", COMMIT, "| code fingerprint:", CODE_FINGERPRINT)

In [ ]:
# ---- 2. dependencies (Kaggle) ------------------------------------------------------
# numpy and torch are NEVER moved: pandas/pyarrow and CUDA in the image are built for them.
if ON_KAGGLE and INSTALL_DEPS:
    import numpy as _np, torch
    _tv = tuple(int(x) for x in torch.__version__.split("+")[0].split(".")[:2])
    pkgs = ["transformers==4.50.0", "datasets==3.4.1", "tokenizers>=0.21,<0.22", "evaluate>=0.4.3",
            "accelerate>=1.4.0", "hydra-core>=1.3.2", "hydra-colorlog>=1.2.0",
            "torchsurv" if _tv >= (2, 8) else "torchsurv==0.1.4",
            "h5py>=3.13", "logdecorator>=2.5", "einops==0.8.1", "torchtuples>=0.2.2",
            "numba", "nvidia-ml-py", "polars", "tqdm", "tabulate",
            "scikit-survival" if int(_np.__version__.split(".")[0]) >= 2 else "scikit-survival==0.22.2",
            "pycox==0.3.0", "peft", "easydict", "lifelines", "osqp", "patsy",
            f"numpy=={_np.__version__}", f"torch=={torch.__version__.split('+')[0]}"]
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *pkgs], check=True)
    # SurvivalEVAL declares torch>=2.13; it runs on older torch, so its pins are skipped
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-deps", "SurvivalEVAL==0.8.5"], check=True)
import torch
print("torch", torch.__version__, "| cuda" if torch.cuda.is_available() else "| cpu")

In [ ]:
# ---- 3. runner ----------------------------------------------------------------------
from IPython.display import Markdown, display
import pandas as pd
from scripts.runner import Runner, select_best
from scripts import v2

RUN_NAME = NAME + ("_smoke" if SMOKE_TEST else "")
r = Runner(repo=REPO, results_dir=RESULTS_ROOT / RUN_NAME,
           workers=WORKERS or (2 if ON_KAGGLE else 4),
           smoke_epochs=SMOKE_EPOCHS if SMOKE_TEST else None, gpu="0")

# Resume: attach this notebook's previous output ("Add Input") and finished runs are
# copied back, so only the missing ones are trained.
n_copied = 0
for root in INPUT_ROOTS:
    for d in root.rglob(RUN_NAME):
        if d.is_dir() and d != r.results_dir:
            for f in d.glob("*.json"):
                if not (r.results_dir / f.name).exists():
                    shutil.copy2(f, r.results_dir / f.name); n_copied += 1
print(f"resumed {n_copied} files from attached inputs")
(r.results_dir / "environment.json").write_text(json.dumps({
    "commit": COMMIT, "code_fingerprint": CODE_FINGERPRINT, "torch": torch.__version__,
    "cuda": torch.cuda.is_available(), "smoke": SMOKE_TEST, "name": RUN_NAME}, indent=2))
print("results ->", r.results_dir)

## Runs

In [ ]:
if SMOKE_TEST:
    SEEDS, DATASETS = SEEDS[:1], DATASETS[:1]
    BACKBONES = {"scratch": BACKBONES["scratch"]}
v2.prepare(r, DATASETS, SEEDS, REPO)
runs = []
for name in DATASETS:
    for bb, bov in BACKBONES.items():
        for obj, oov in OBJECTIVES.items():
            runs += v2.runs("llm", name, SEEDS, REPO, extra=bov + oov + [f"llm_epochs={LLM_EPOCHS}"],
                            tag=f"llm_{bb}_{obj}", info={"stage": "llm", "backbone": bb, "objective": obj})
print(len(runs), "runs")
# one LLM at a time: two 0.5B models do not fit next to each other on a T4
r.workers = 1
r.run_many(runs, time_budget_min=TIME_BUDGET_MIN)

## Tables

In [ ]:
# ---- tables + zip ---------------------------------------------------------------------
rep = r.results_dir / "report"
rep.mkdir(exist_ok=True)
res = r.collect()
res.to_csv(rep / "all_runs.csv", index=False)
if len(res):
    t = v2.table(res, ["dataset", "model", "variant"])
    t.to_csv(rep / "summary.csv", index=False)
    display(t)
print("zip:", r.zip())

## Clean up

In [ ]:
if ON_KAGGLE and CLEAN_UP_AT_END:
    shutil.rmtree(REPO, ignore_errors=True)
    print("removed", REPO, "- results are in", r.results_dir)